In [2]:
"""
==========================================================
AI vs Real Image Detection Pipeline v4 — COMPLETE ROBUST VERSION
==========================================================

Key Changes from v3:
1. REMOVED metadata leakage features (resolution, aspect ratio, etc.)
2. ADDED enhanced multi-prompt CLIP ensemble (4 features vs 2)
3. ADDED SRM noise filter detector (5 features)
4. ADDED Patch consistency detector (4 features)
5. ADDED JPEG ghost detector (3 features)
6. OVERHAULED training: stacking, calibration, stronger regularization
7. ADDED confidence-aware prediction system

Label Convention:
  Real = 0, AI = 1
"""


'\n==========================================================\nAI vs Real Image Detection Pipeline v4 — COMPLETE ROBUST VERSION\n==========================================================\n\nKey Changes from v3:\n1. REMOVED metadata leakage features (resolution, aspect ratio, etc.)\n2. ADDED enhanced multi-prompt CLIP ensemble (4 features vs 2)\n3. ADDED SRM noise filter detector (5 features)\n4. ADDED Patch consistency detector (4 features)\n5. ADDED JPEG ghost detector (3 features)\n6. OVERHAULED training: stacking, calibration, stronger regularization\n7. ADDED confidence-aware prediction system\n\nLabel Convention:\n  Real = 0, AI = 1\n'

## CELL 1: Configuration


In [3]:
import os
from pathlib import Path

# 1. INPUT DIRECTORY
INPUT_DIR = "/kaggle/input/datasets/ishu15m/ai-vs-real-images"
DATASET = Path(INPUT_DIR)

# 2. OUTPUT DIRECTORY
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. CSV FILES
REAL_CSV = os.path.join(OUTPUT_DIR, "real_detector_dataset_v4.csv")
AI_CSV = os.path.join(OUTPUT_DIR, "ai_detector_dataset_v4.csv")
COMBINED_CSV = os.path.join(OUTPUT_DIR, "combined_detector_dataset_v4.csv")

# Storage folders
CORRECT_REAL_DIR = os.path.join(OUTPUT_DIR, "correctly_classified_v4/real")
CORRECT_AI_DIR = os.path.join(OUTPUT_DIR, "correctly_classified_v4/ai")
os.makedirs(CORRECT_REAL_DIR, exist_ok=True)
os.makedirs(CORRECT_AI_DIR, exist_ok=True)

# Model output
MODEL_DIR = os.path.join(OUTPUT_DIR, "models_v4")
os.makedirs(MODEL_DIR, exist_ok=True)

PROJECT_DIR = OUTPUT_DIR

# Auto-delete old CSVs
for csv_file in [REAL_CSV, AI_CSV, COMBINED_CSV]:
    if os.path.exists(csv_file):
        os.remove(csv_file)
        print(f"Deleted old CSV: {csv_file}")

## CELL 2: Download Dataset


In [4]:
import kagglehub

path = kagglehub.dataset_download("ishu15m/ai-vs-real-images")
print("Path to dataset files:", path)
DATASET = Path(path)

print("\nDataset structure:")
for item in sorted(DATASET.rglob("*")):
    if item.is_dir():
        count = sum(1 for f in item.iterdir() if f.is_file())
        print(f"  [DIR]  {item.relative_to(DATASET)}  ({count} files)")

Path to dataset files: /kaggle/input/datasets/ishu15m/ai-vs-real-images

Dataset structure:
  [DIR]  AI-images  (0 files)
  [DIR]  AI-images/AI-images  (0 files)
  [DIR]  AI-images/AI-images/ai_animals  (50 files)
  [DIR]  AI-images/AI-images/ai_buildings  (50 files)
  [DIR]  AI-images/AI-images/ai_food  (31 files)
  [DIR]  AI-images/AI-images/ai_human  (35 files)
  [DIR]  AI-images/AI-images/ai_interior  (33 files)
  [DIR]  AI-images/AI-images/ai_items  (34 files)
  [DIR]  AI-images/AI-images/ai_nature  (45 files)
  [DIR]  Real-images  (0 files)
  [DIR]  Real-images/Real-images  (0 files)
  [DIR]  Real-images/Real-images/real_animals  (50 files)
  [DIR]  Real-images/Real-images/real_buildings  (53 files)
  [DIR]  Real-images/Real-images/real_food  (33 files)
  [DIR]  Real-images/Real-images/real_humans  (50 files)
  [DIR]  Real-images/Real-images/real_interior  (28 files)
  [DIR]  Real-images/Real-images/real_items  (34 files)
  [DIR]  Real-images/Real-images/real_nature  (50 files)


## CELL 3: Metadata Extraction


In [5]:
from pathlib import Path
from PIL import Image
import pandas as pd
import os

all_dataset_folders = list(DATASET.iterdir())
print("Scanning primary dataset only for metadata.")

meta_rows = []
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".avif", ".tif", ".tiff"}

for folder in all_dataset_folders:
    if not folder.is_dir():
        continue
    label = folder.name
    for img_path in folder.rglob("*"):
        if img_path.suffix.lower() not in VALID_EXTENSIONS:
            continue
        try:
            with Image.open(img_path) as img:
                width, height = img.size
                img_format = img.format
                meta_rows.append({
                    "image_id": img_path.name, "label": label,
                    "source": "primary", "width": width,
                    "height": height, "format": img_format,
                })
        except Exception as e:
            print(f"Skipped {img_path.name}: {e}")

meta_df = pd.DataFrame(meta_rows)
meta_output = os.path.join(PROJECT_DIR, "image_metadata_v4.csv")
meta_df.to_csv(meta_output, index=False)
print(f"\nSaved metadata for {len(meta_df)} images to {meta_output}")
print(meta_df.head())

Scanning primary dataset only for metadata.

Saved metadata for 576 images to /kaggle/working/image_metadata_v4.csv
    image_id        label   source  width  height format
0   (2).jpeg  Real-images  primary    225     225   JPEG
1  (12).jpeg  Real-images  primary    173     291   JPEG
2  (33).jpeg  Real-images  primary    194     259   JPEG
3  (23).jpeg  Real-images  primary    213     237   JPEG
4  (15).jpeg  Real-images  primary    225     225   JPEG


## CELL 4: Convert to PNG


In [6]:
from PIL import Image
from tqdm import tqdm

SOURCE = DATASET
DESTINATION = Path(os.path.join(PROJECT_DIR, "png_dataset"))
DESTINATION.mkdir(parents=True, exist_ok=True)
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".avif"}

all_files = [f for f in SOURCE.rglob("*") if f.suffix.lower() in VALID_EXTENSIONS]
print(f"Found {len(all_files)} images to convert")

for file in tqdm(all_files, desc="Converting to PNG"):
    relative_path = file.relative_to(SOURCE)
    output_file = DESTINATION / relative_path.with_suffix(".png")
    output_file.parent.mkdir(parents=True, exist_ok=True)
    try:
        img = Image.open(file)
        exif_data = img.info.get("exif", None)
        img = img.convert("RGB")
        if exif_data:
            img.save(output_file, format="PNG", exif=exif_data)
        else:
            img.save(output_file, format="PNG")
    except Exception as e:
        print(f"Error: {file} -- {e}")
print("Conversion complete.")

Found 576 images to convert


Converting to PNG: 100%|██████████| 576/576 [07:46<00:00,  1.24it/s]

Conversion complete.


## CELL 5: Remove Duplicates


In [7]:
import imagehash

PNG_DATASET = Path(os.path.join(PROJECT_DIR, "png_dataset"))
folders = [p for p in PNG_DATASET.iterdir() if p.is_dir()]

for folder in folders:
    print(f"\n{'='*80}\nProcessing: {folder.name}\n{'='*80}")
    hash_dict = {}
    duplicates_found = 0
    for img_path in folder.rglob("*"):
        if img_path.suffix.lower() != ".png":
            continue
        try:
            with Image.open(img_path) as img:
                phash = imagehash.phash(img)
            if phash in hash_dict:
                print(f"[DUPLICATE] Kept: {hash_dict[phash].name} | Deleted: {img_path.name}")
                img_path.unlink()
                duplicates_found += 1
            else:
                hash_dict[phash] = img_path
        except Exception as e:
            print(f"Error: {img_path}: {e}")
    print(f"Total duplicates removed: {duplicates_found}")


Processing: AI-images
[DUPLICATE] Kept: (16).png | Deleted: (22).png
[DUPLICATE] Kept: (16).png | Deleted: (18).png
[DUPLICATE] Kept: (24).png | Deleted: (23).png
Total duplicates removed: 3

Processing: Real-images
Total duplicates removed: 0


## CELL 6: Verify Counts


In [8]:
print("Image counts by folder:")
print("=" * 50)
for folder in sorted(PNG_DATASET.iterdir()):
    if folder.is_dir():
        count = len(list(folder.rglob("*.png")))
        print(f"  {folder.name:30s} : {count:6d}")
total = len(list(PNG_DATASET.rglob("*.png")))
print(f"{'':30s}   {'---':>6}")
print(f"  {'TOTAL':30s} : {total:6d}")

Image counts by folder:
  AI-images                      :    257
  Real-images                    :    269
                                    ---
  TOTAL                          :    526


## CELL 7: Transformations (44 unified — same as v3)


In [9]:
from PIL import Image, ImageEnhance, ImageFilter
from io import BytesIO
import numpy as np
import cv2

def jpeg_compress(img, quality):
    buf = BytesIO(); img.save(buf, format="JPEG", quality=quality); buf.seek(0)
    r = Image.open(buf).convert("RGB"); r.load(); return r

def gaussian_blur(img, radius):
    return img.filter(ImageFilter.GaussianBlur(radius))

def sharpen(img, factor):
    return ImageEnhance.Sharpness(img).enhance(factor)

def brightness(img, factor):
    return ImageEnhance.Brightness(img).enhance(factor)

def contrast(img, factor):
    return ImageEnhance.Contrast(img).enhance(factor)

def gaussian_noise(img, sigma):
    a = np.array(img).astype(np.float32)
    return Image.fromarray(np.clip(a + np.random.normal(0, sigma, a.shape), 0, 255).astype(np.uint8))

def rotate(img, angle):
    return img.rotate(angle, expand=False, fillcolor=(128,128,128))

def horizontal_flip(img):
    return img.transpose(Image.FLIP_LEFT_RIGHT)

def hue_shift(img, shift):
    hsv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2HSV)
    hsv[:,:,0] = (hsv[:,:,0].astype(int) + shift) % 180
    return Image.fromarray(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB))

def saturation(img, factor):
    return ImageEnhance.Color(img).enhance(factor)

def resize_scale(img, scale):
    w,h = img.size; return img.resize((max(1,int(w*scale)),max(1,int(h*scale))))

def center_crop(img, pct):
    w,h = img.size; nw,nh = int(w*pct),int(h*pct)
    l,t = (w-nw)//2,(h-nh)//2; return img.crop((l,t,l+nw,t+nh))

def screenshot_phone(img):
    w,h = img.size; img = img.resize((1080,max(1,int(h*(1080/max(1,w))))))
    buf = BytesIO(); img.save(buf, format="PNG"); buf.seek(0)
    r = Image.open(buf).convert("RGB"); r.load(); return r

def screenshot_social(img):
    w,h = img.size; img = img.resize((1080,max(1,int(h*(1080/max(1,w))))))
    buf = BytesIO(); img.save(buf, format="JPEG", quality=85); buf.seek(0)
    r = Image.open(buf).convert("RGB"); r.load(); return r

def screenshot_messaging(img):
    w,h = img.size; img = img.resize((720,max(1,int(h*(720/max(1,w))))))
    buf = BytesIO(); img.save(buf, format="JPEG", quality=70); buf.seek(0)
    r = Image.open(buf).convert("RGB"); r.load(); return r

def motion_blur(image, kernel_size=15, angle=45):
    a = np.array(image)
    M = cv2.getRotationMatrix2D((kernel_size/2,kernel_size/2), angle, 1)
    k = np.diag(np.ones(kernel_size))
    k = cv2.warpAffine(k, M, (kernel_size,kernel_size)) / kernel_size
    return Image.fromarray(cv2.filter2D(a, -1, k))

def chromatic_aberration(image, shift=3):
    a = np.array(image); r,g,b = cv2.split(a); rows,cols = r.shape
    r_s = cv2.warpAffine(r, np.float32([[1,0,shift],[0,1,0]]), (cols,rows))
    b_s = cv2.warpAffine(b, np.float32([[1,0,-shift],[0,1,0]]), (cols,rows))
    return Image.fromarray(cv2.merge((r_s, g, b_s)))

def poisson_noise(image):
    a = np.array(image)/255.0
    return Image.fromarray((np.clip(np.random.poisson(a*255)/255,0,1)*255).astype(np.uint8))

def webp_compression(image, quality=50):
    buf = BytesIO(); image.save(buf, format="WEBP", quality=quality); buf.seek(0)
    r = Image.open(buf).convert("RGB"); r.load(); return r

def cutout_simulation(image, size=50):
    a = np.array(image); h,w,_ = a.shape
    if h>size and w>size:
        y,x = np.random.randint(0,h-size), np.random.randint(0,w-size)
        a[y:y+size,x:x+size] = 0
    return Image.fromarray(a)

def mixed_degradation(image):
    img = motion_blur(image.copy(), kernel_size=9, angle=15)
    img = poisson_noise(img)
    buf = BytesIO(); img.save(buf, format="JPEG", quality=60); buf.seek(0)
    r = Image.open(buf).convert("RGB"); r.load(); return r

transformations = {
    "none": lambda x: x,
    "jpeg_90": lambda x: jpeg_compress(x,90), "jpeg_70": lambda x: jpeg_compress(x,70),
    "jpeg_50": lambda x: jpeg_compress(x,50),
    "blur_2": lambda x: gaussian_blur(x,2), "blur_4": lambda x: gaussian_blur(x,4),
    "blur_6": lambda x: gaussian_blur(x,6),
    "sharp_1.5": lambda x: sharpen(x,1.5), "sharp_2": lambda x: sharpen(x,2),
    "sharp_3": lambda x: sharpen(x,3),
    "bright_0.7": lambda x: brightness(x,0.7), "bright_1.3": lambda x: brightness(x,1.3),
    "bright_1.6": lambda x: brightness(x,1.6),
    "contrast_0.7": lambda x: contrast(x,0.7), "contrast_1.3": lambda x: contrast(x,1.3),
    "contrast_1.6": lambda x: contrast(x,1.6),
    "noise_5": lambda x: gaussian_noise(x,5), "noise_15": lambda x: gaussian_noise(x,15),
    "noise_30": lambda x: gaussian_noise(x,30),
    "rotate_5": lambda x: rotate(x,5), "rotate_15": lambda x: rotate(x,15),
    "rotate_30": lambda x: rotate(x,30),
    "flip": lambda x: horizontal_flip(x),
    "hue_10": lambda x: hue_shift(x,10), "hue_30": lambda x: hue_shift(x,30),
    "hue_60": lambda x: hue_shift(x,60),
    "sat_0.7": lambda x: saturation(x,0.7), "sat_1.3": lambda x: saturation(x,1.3),
    "sat_1.8": lambda x: saturation(x,1.8),
    "resize_75": lambda x: resize_scale(x,0.75), "resize_50": lambda x: resize_scale(x,0.50),
    "resize_25": lambda x: resize_scale(x,0.25),
    "crop_95": lambda x: center_crop(x,0.95), "crop_85": lambda x: center_crop(x,0.85),
    "crop_70": lambda x: center_crop(x,0.70),
    "screenshot_phone": lambda x: screenshot_phone(x),
    "screenshot_social": lambda x: screenshot_social(x),
    "screenshot_messaging": lambda x: screenshot_messaging(x),
    "motion_blur_15": lambda x: motion_blur(x,15,45),
    "chromatic_aberration_3": lambda x: chromatic_aberration(x,3),
    "poisson_noise": lambda x: poisson_noise(x),
    "webp_50": lambda x: webp_compression(x,50),
    "cutout_50": lambda x: cutout_simulation(x,50),
    "mixed_degradation": lambda x: mixed_degradation(x),
}
print(f"{len(transformations)} transformations loaded.")

44 transformations loaded.


## CELL 8-9: GPU Detectors — SigLIP + ViT (UNCHANGED from v3)


In [10]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- SigLIP ---
siglip_model_name = "Ateeqq/ai-vs-human-image-detector"
siglip_processor = AutoImageProcessor.from_pretrained(siglip_model_name)
siglip_model = AutoModelForImageClassification.from_pretrained(siglip_model_name).to(device)
_siglip_id2label = siglip_model.config.id2label
_siglip_ai_idx = None
for idx, label in _siglip_id2label.items():
    if any(kw in str(label).lower() for kw in ["ai","fake","generated","artificial"]):
        _siglip_ai_idx = int(idx); break
if _siglip_ai_idx is None: _siglip_ai_idx = 1
print(f"  SigLIP AI class index = {_siglip_ai_idx}")

def siglip_batch_detector(images_list):
    if not images_list: return []
    inputs = siglip_processor(images=images_list, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad(): outputs = siglip_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    return [{"siglip_ai_prob": float(probs[i, _siglip_ai_idx]),
             "siglip_confidence": float(probs[i].max())} for i in range(len(images_list))]

# --- ViT ---
vit_model_name = "dima806/ai_vs_human_generated_image_detection"
vit_processor = AutoImageProcessor.from_pretrained(vit_model_name)
vit_model = AutoModelForImageClassification.from_pretrained(vit_model_name).to(device)
_vit_id2label = vit_model.config.id2label
_vit_ai_idx = None
for idx, label in _vit_id2label.items():
    if any(kw in str(label).lower() for kw in ["ai","fake","generated","artificial"]):
        _vit_ai_idx = int(idx); break
if _vit_ai_idx is None: _vit_ai_idx = 1
print(f"  ViT AI class index = {_vit_ai_idx}")

def vit_batch_detector(images_list):
    if not images_list: return []
    inputs = vit_processor(images=images_list, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad(): outputs = vit_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    return [{"vit_ai_prob": float(probs[i, _vit_ai_idx]),
             "vit_confidence": float(probs[i].max())} for i in range(len(images_list))]

print("SigLIP + ViT detectors loaded.")

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

...

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/737 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  ViT AI class index = 1
SigLIP + ViT detectors loaded.


## CELL 10: CLIP ENHANCED Multi-Prompt Detector [v4 NEW]


In [11]:
from transformers import CLIPProcessor, CLIPModel

clip_model_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)

_clip_ai_prompts = [
    "an AI generated image", "a synthetic image created by artificial intelligence",
    "a computer generated digital artwork", "an image generated by a neural network",
    "a deepfake or AI-manipulated image", "a digitally rendered synthetic photograph",
    "an artificially created image with perfect details", "a machine learning generated picture",
]
_clip_real_prompts = [
    "a real photograph taken by a camera", "a natural photograph of the real world",
    "an authentic unedited photograph", "a genuine camera captured image",
    "a real photo taken by a human photographer", "a candid photograph of a real scene",
    "an unmanipulated real world photograph", "a true photographic image captured with a lens",
]

def clip_batch_detector(images_list):
    if not images_list: return []
    results = []
    batch_size = 8
    for start in range(0, len(images_list), batch_size):
        batch = images_list[start:start+batch_size]
        all_probs = []
        for ai_prompt, real_prompt in zip(_clip_ai_prompts, _clip_real_prompts):
            inputs = clip_processor(text=[real_prompt, ai_prompt], images=batch,
                                    return_tensors="pt", padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad(): outputs = clip_model(**inputs)
            probs = torch.softmax(outputs.logits_per_image, dim=1)
            all_probs.append([float(probs[i, 1]) for i in range(len(batch))])
        all_probs_np = np.array(all_probs)
        for i in range(len(batch)):
            pp = all_probs_np[:, i]
            results.append({
                "clip_ai_prob": float(np.mean(pp)),
                "clip_ai_prob_std": float(np.std(pp)),
                "clip_ai_prob_max": float(np.max(pp)),
                "clip_confidence": abs(float(np.mean(pp)) - 0.5) * 2,
            })
    return results

print(f"CLIP multi-prompt detector loaded ({len(_clip_ai_prompts)} prompt pairs). [v4]")


# ==========================================================
# CELL 11-22: CPU Detectors (FFT, ELA, Noise, DCT, Wavelet,
# ColorHist, LBP, Edge, PixelStats, GAN, Gradient — UNCHANGED)
# + NEW: SafeMetadata, SRM, PatchConsistency, JPEGGhost
# ==========================================================

# --- FFT ---
def fft_detector(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY).astype(np.float32)
    h,w = gray.shape
    mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray))))
    cy,cx = h//2,w//2
    Y,X = np.ogrid[:h,:w]
    r = np.sqrt((X-cx)**2+(Y-cy)**2)
    r_max = np.sqrt(cx**2+cy**2)+1e-10; r_norm = r/r_max
    lo,mid,hi = r_norm<=0.33,(r_norm>0.33)&(r_norm<=0.66),r_norm>0.66
    le = float(np.mean(mag[lo])) if lo.any() else 0.0
    me = float(np.mean(mag[mid])) if mid.any() else 0.0
    he = float(np.mean(mag[hi])) if hi.any() else 0.0
    te = float(np.sum(mag))+1e-10
    mf = mag.flatten(); mp = mf/(mf.sum()+1e-10)
    return {"fft_low_energy":le,"fft_mid_energy":me,"fft_high_energy":he,
            "fft_high_freq_ratio":float(np.sum(mag[hi])/te),
            "fft_entropy":float(-np.sum(mp*np.log(mp+1e-10))),
            "fft_mid_to_high_ratio":float(me/(he+1e-10))}

# --- ELA ---
def ela_detector(image):
    img_rgb = image.convert("RGB")
    buf95 = BytesIO(); img_rgb.save(buf95,format="JPEG",quality=95); buf95.seek(0)
    ela95 = np.array(ImageChops.difference(img_rgb,Image.open(buf95).convert("RGB"))).astype(np.float32)
    buf75 = BytesIO(); img_rgb.save(buf75,format="JPEG",quality=75); buf75.seek(0)
    ela75 = np.array(ImageChops.difference(img_rgb,Image.open(buf75).convert("RGB"))).astype(np.float32)
    s = float(np.std(ela95))+1e-8
    return {"ela_mean_q95":float(np.mean(ela95)),"ela_std_q95":float(np.std(ela95)),
            "ela_max_q95":float(np.max(ela95)),"ela_mean_q75":float(np.mean(ela75)),
            "ela_std_q75":float(np.std(ela75)),
            "ela_skew":float(np.mean(((ela95-np.mean(ela95))/s)**3)),
            "ela_kurtosis":float(np.mean(((ela95-np.mean(ela95))/s)**4))}

from PIL import ImageChops

# --- Noise ---
def noise_detector(image):
    img = np.array(image).astype(np.float32)
    rg = img - cv2.GaussianBlur(img,(5,5),0)
    gray = cv2.cvtColor(img.astype(np.uint8),cv2.COLOR_RGB2GRAY).astype(np.float32)
    rm = gray - cv2.medianBlur(gray.astype(np.uint8),5).astype(np.float32)
    lap = cv2.Laplacian(gray, cv2.CV_32F)
    cs = [float(np.std(rg[:,:,c])) for c in range(min(3,img.shape[2] if img.ndim==3 else 1))]
    cs = cs + [0.0]*(3-len(cs))
    return {"noise_std_gauss":float(np.std(rg)),"noise_mean_gauss":float(np.mean(np.abs(rg))),
            "noise_std_median":float(np.std(rm)),"noise_laplacian_var":float(np.var(lap)),
            "noise_channel_std_range":float(max(cs)-min(cs)),"noise_channel_std_mean":float(np.mean(cs))}

# --- SAFE Metadata [v4 REWRITTEN] ---
def metadata_detector(image):
    has_icc = int(image.info.get("icc_profile") is not None)
    exif = image.getexif() if hasattr(image,'getexif') else {}
    efc = len(exif) if exif else 0
    hcm = int(271 in exif) if exif else 0
    hgps = int(34853 in exif) if exif else 0
    w,h = image.size; tp = w*h
    try:
        buf = BytesIO(); image.save(buf,format="PNG"); cr = float(buf.tell()/max(tp*3,1))
    except: cr = 0.0
    mode = image.mode
    has_alpha = int("A" in mode)
    num_channels = len(mode)
    return {"meta_has_icc_profile":has_icc,"meta_exif_field_count":float(min(efc,100)),
            "meta_has_camera_make":hcm,"meta_has_gps":hgps,"meta_compression_ratio":cr,
            "meta_has_alpha":has_alpha, "meta_num_channels":num_channels}

# --- DCT ---
def dct_detector(image):
    gray = cv2.cvtColor(np.array(image),cv2.COLOR_RGB2GRAY).astype(np.float32)
    h,w = gray.shape; bs = 8; hb,wb = h//bs,w//bs
    if hb<2 or wb<2: return {"dct_block_energy":0.0,"dct_block_std":0.0,"dct_boundary_strength":0.0,"dct_hf_coeff_ratio":0.0}
    de = np.array([float(np.sum(np.abs(cv2.dct(gray[by*bs:(by+1)*bs,bx*bs:(bx+1)*bs])[4:,4:])))
                   for by in range(hb) for bx in range(wb)])
    bd = np.array([float(np.mean(np.abs(gray[(by+1)*bs-1,bx*bs:(bx+1)*bs]-gray[(by+1)*bs,bx*bs:(bx+1)*bs])))
                   for by in range(hb-1) for bx in range(wb)]) if hb>1 else np.array([0.0])
    return {"dct_block_energy":float(np.mean(de)),"dct_block_std":float(np.std(de)),
            "dct_boundary_strength":float(np.mean(bd)),
            "dct_hf_coeff_ratio":float(np.sum(de>np.median(de))/max(len(de),1))}

# --- Wavelet ---
def wavelet_detector(image):
    gray = cv2.cvtColor(np.array(image),cv2.COLOR_RGB2GRAY).astype(np.float32)
    h,w = gray.shape; h2,w2 = h-h%2,w-w%2
    if h2<4 or w2<4: return {"wavelet_detail_energy":0.0,"wavelet_approx_energy":0.0,"wavelet_detail_ratio":0.0,"wavelet_hh_entropy":0.0,"wavelet_hh_std":0.0}
    img = gray[:h2,:w2]
    lo = (img[0::2,:]+img[1::2,:])/2; hv = (img[0::2,:]-img[1::2,:])/2
    LL = (lo[:,0::2]+lo[:,1::2])/2; LH = (lo[:,0::2]-lo[:,1::2])/2
    HL = (hv[:,0::2]+hv[:,1::2])/2; HH = (hv[:,0::2]-hv[:,1::2])/2
    de = float(np.mean(np.abs(LH))+np.mean(np.abs(HL))+np.mean(np.abs(HH)))
    ae = float(np.mean(np.abs(LL)))+1e-10
    hf = np.abs(HH).flatten(); hp = hf/(hf.sum()+1e-10)
    return {"wavelet_detail_energy":de,"wavelet_approx_energy":ae,"wavelet_detail_ratio":de/ae,
            "wavelet_hh_entropy":float(-np.sum(hp*np.log(hp+1e-10))),"wavelet_hh_std":float(np.std(HH))}

# --- Color Histogram ---
def color_histogram_detector(image):
    img = np.array(image)
    ents = []
    for c in range(3):
        h,_ = np.histogram(img[:,:,c],bins=256,range=(0,256))
        h = h.astype(np.float64)/(h.sum()+1e-10)
        ents.append(float(-np.sum(h*np.log(h+1e-10))))
    r,g,b = img[:,:,0].flatten().astype(np.float64),img[:,:,1].flatten().astype(np.float64),img[:,:,2].flatten().astype(np.float64)
    rg = float(np.corrcoef(r,g)[0,1]) if len(r)>1 else 0.0
    rb = float(np.corrcoef(r,b)[0,1]) if len(r)>1 else 0.0
    if np.isnan(rg): rg = 0.0
    if np.isnan(rb): rb = 0.0
    return {"color_entropy_r":ents[0],"color_entropy_g":ents[1],"color_entropy_b":ents[2],"color_corr_rg":rg,"color_corr_rb":rb}

# --- LBP ---
def lbp_detector(image):
    gray = cv2.cvtColor(np.array(image),cv2.COLOR_RGB2GRAY)
    if max(gray.shape)>512:
        s = 512.0/max(gray.shape); gray = cv2.resize(gray,None,fx=s,fy=s,interpolation=cv2.INTER_AREA)
    h,w = gray.shape
    if h<3 or w<3: return {"lbp_entropy":0.0,"lbp_uniformity":0.0,"lbp_mean":0.0,"lbp_std":0.0}
    c = gray[1:-1,1:-1].astype(np.int16)
    lbp = np.zeros_like(c,dtype=np.uint8)
    for bit,(dy,dx) in enumerate([(0,0),(0,1),(0,2),(1,2),(2,2),(2,1),(2,0),(1,0)]):
        lbp |= (gray[dy:dy+h-2,dx:dx+w-2].astype(np.int16) >= c).astype(np.uint8) << (7-bit)
    hist,_ = np.histogram(lbp,bins=256,range=(0,256))
    hist = hist.astype(np.float64)/(hist.sum()+1e-10)
    return {"lbp_entropy":float(-np.sum(hist*np.log(hist+1e-10))),"lbp_uniformity":float(np.sum(hist**2)),
            "lbp_mean":float(np.mean(lbp.astype(np.float32))),"lbp_std":float(np.std(lbp.astype(np.float32)))}

# --- Edge Coherence ---
def edge_coherence_detector(image):
    gray = cv2.cvtColor(np.array(image),cv2.COLOR_RGB2GRAY)
    if max(gray.shape)>512:
        s = 512.0/max(gray.shape); gray = cv2.resize(gray,None,fx=s,fy=s,interpolation=cv2.INTER_AREA)
    ed = float(np.mean(cv2.Canny(gray,50,150)>0))
    gx = cv2.Sobel(gray.astype(np.float32),cv2.CV_32F,1,0,ksize=3)
    gy = cv2.Sobel(gray.astype(np.float32),cv2.CV_32F,0,1,ksize=3)
    mag = np.sqrt(gx**2+gy**2); d = np.arctan2(gy,gx)
    sm = mag > np.percentile(mag,75)
    if sm.sum()>10:
        dh,_ = np.histogram(d[sm],bins=36,range=(-np.pi,np.pi))
        dh = dh.astype(np.float64)/(dh.sum()+1e-10)
        de = float(-np.sum(dh*np.log(dh+1e-10))); du = float(np.std(dh))
    else: de,du = 0.0,0.0
    return {"edge_density":ed,"edge_dir_entropy":de,"edge_dir_uniformity":du,"edge_magnitude_std":float(np.std(mag))}

# --- Pixel Stats ---
def pixel_stats_detector(image):
    img = np.array(image).astype(np.float32)
    gray = cv2.cvtColor(img.astype(np.uint8),cv2.COLOR_RGB2GRAY)
    nz = gray[gray>0].flatten().astype(np.float64)
    bd = 0.0
    if len(nz)>100:
        fd = (nz/(10**np.floor(np.log10(nz+1e-10)))).astype(int)
        fd = fd[(fd>=1)&(fd<=9)]
        if len(fd)>0:
            dc = np.bincount(fd,minlength=10)[1:]
            dd = dc.astype(np.float64)/(dc.sum()+1e-10)
            bd = float(np.sum(np.abs(dd-np.log10(1+1/np.arange(1,10)))))
    h,_ = np.histogram(gray,bins=256,range=(0,256)); h = h.astype(np.float64)/(h.sum()+1e-10)
    pe = float(-np.sum(h*np.log(h+1e-10)))
    tp = img.shape[0]*img.shape[1]
    if img.ndim==3 and img.shape[2]>=3:
        pq = (img[:,:,:3]//8).astype(np.uint8)
        codes = pq[:,:,0].astype(np.int32)*1024+pq[:,:,1].astype(np.int32)*32+pq[:,:,2].astype(np.int32)
        uc = len(np.unique(codes))
    else: uc = len(np.unique(gray))
    return {"pixel_benford_dev":bd,"pixel_entropy":pe,"pixel_unique_ratio":float(uc/max(tp,1)),
            "pixel_dynamic_range":float(np.max(gray)-np.min(gray)),"pixel_mean_brightness":float(np.mean(gray))}

# --- GAN Fingerprint ---
def gan_fingerprint_detector(image):
    gray = cv2.cvtColor(np.array(image),cv2.COLOR_RGB2GRAY).astype(np.float32)
    if max(gray.shape)>256:
        s = 256.0/max(gray.shape); gray = cv2.resize(gray,None,fx=s,fy=s,interpolation=cv2.INTER_AREA)
    h,w = gray.shape
    if h<8 or w<8: return {"gan_autocorr_peak":0.0,"gan_autocorr_mean":0.0,"gan_periodicity":0.0,"gan_spectral_flatness":0.0}
    psd = np.abs(np.fft.fft2(gray))**2
    ac = np.fft.fftshift(np.fft.ifft2(psd).real)
    cv = ac[h//2,w//2]
    acn = ac/cv if cv>0 else ac
    cy,cx = h//2,w//2; acm = acn.copy(); r = 3
    acm[max(0,cy-r):min(h,cy+r+1),max(0,cx-r):min(w,cx+r+1)] = 0.0
    rp = acn[cy,cx+1:]
    ps = 0.0
    if len(rp)>10:
        d = np.diff(np.sign(np.diff(rp))); ps = float(len(np.where(d<0)[0])/max(len(rp),1))
    pf = psd.flatten()+1e-10
    sf = float(np.exp(np.mean(np.log(pf)))/(np.mean(pf)+1e-10))
    return {"gan_autocorr_peak":float(np.max(np.abs(acm))),"gan_autocorr_mean":float(np.mean(np.abs(acm))),
            "gan_periodicity":ps,"gan_spectral_flatness":sf}

# --- Gradient ---
def gradient_detector(image):
    gray = cv2.cvtColor(np.array(image),cv2.COLOR_RGB2GRAY).astype(np.float32)
    gx = cv2.Sobel(gray,cv2.CV_32F,1,0,ksize=3); gy = cv2.Sobel(gray,cv2.CV_32F,0,1,ksize=3)
    mag = np.sqrt(gx**2+gy**2); gm = float(np.mean(mag)); gs = float(np.std(mag))+1e-10
    gk = float(np.mean((mag-gm)**4)/(gs**4+1e-10))
    md = float(np.median(mag)); hr = float(np.mean(mag>2*md)) if md>0 else 0.0
    return {"gradient_mean":gm,"gradient_std":gs,"gradient_kurtosis":gk,"gradient_high_ratio":hr}


# --- SRM Noise Filters [v4 NEW] ---
def srm_detector(image):
    img = np.array(image).astype(np.float32)
    if max(img.shape[:2])>512:
        s = 512.0/max(img.shape[:2]); img = cv2.resize(img,None,fx=s,fy=s,interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(img.astype(np.uint8),cv2.COLOR_RGB2GRAY).astype(np.float32)
    k1 = np.array([[0,0,0],[0,-1,1],[0,0,0]],dtype=np.float32)
    k2 = np.array([[0,0,0],[1,-2,1],[0,0,0]],dtype=np.float32)
    k3 = np.array([[0,-1,0],[-1,4,-1],[0,-1,0]],dtype=np.float32)
    combined = np.abs(cv2.filter2D(gray,cv2.CV_32F,k1))+np.abs(cv2.filter2D(gray,cv2.CV_32F,k2))+np.abs(cv2.filter2D(gray,cv2.CV_32F,k3))
    se = float(np.mean(combined)); ss = float(np.std(combined))+1e-10
    c = combined-np.mean(combined)
    sk = float(np.mean(c**3)/(ss**3+1e-10)); ku = float(np.mean(c**4)/(ss**4+1e-10))
    cc = 0.0
    if img.ndim==3 and img.shape[2]>=3:
        rn = cv2.filter2D(img[:,:,0],cv2.CV_32F,k3).flatten()
        gn = cv2.filter2D(img[:,:,1],cv2.CV_32F,k3).flatten()
        bn = cv2.filter2D(img[:,:,2],cv2.CV_32F,k3).flatten()
        rg = float(np.corrcoef(rn,gn)[0,1]); rb = float(np.corrcoef(rn,bn)[0,1])
        if np.isnan(rg): rg = 0.0
        if np.isnan(rb): rb = 0.0
        cc = (abs(rg)+abs(rb))/2
    return {"srm_noise_energy":se,"srm_noise_std":ss,"srm_noise_skewness":sk,"srm_noise_kurtosis":ku,"srm_cross_channel_corr":cc}

# --- Patch Consistency [v4 NEW] ---
def patch_consistency_detector(image):
    img = np.array(image).astype(np.float32)
    gray = cv2.cvtColor(img.astype(np.uint8),cv2.COLOR_RGB2GRAY).astype(np.float32)
    h,w = gray.shape; ps = 64
    if h<ps*3 or w<ps*3:
        return {"patch_fft_var":0.0,"patch_noise_var":0.0,"patch_sharpness_var":0.0,"patch_saturation_var":0.0}
    hp,wp = h//ps,w//ps
    hsv = cv2.cvtColor(img.astype(np.uint8),cv2.COLOR_RGB2HSV).astype(np.float32)
    fe,nl,sl,sal = [],[],[],[]
    for py in range(hp):
        for px in range(wp):
            y0,y1,x0,x1 = py*ps,(py+1)*ps,px*ps,(px+1)*ps
            pg = gray[y0:y1,x0:x1]; ph = hsv[y0:y1,x0:x1]
            mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(pg))))
            cy,cx = ps//2,ps//2; Y,X = np.ogrid[:ps,:ps]
            hm = np.sqrt((X-cx)**2+(Y-cy)**2)>(ps*0.33)
            fe.append(float(np.mean(mag[hm])) if hm.any() else 0.0)
            lap = cv2.Laplacian(pg,cv2.CV_32F)
            nl.append(float(np.var(lap))); sl.append(float(np.mean(np.abs(lap))))
            sal.append(float(np.mean(ph[:,:,1])))
    return {"patch_fft_var":float(np.var(fe)),"patch_noise_var":float(np.var(nl)),
            "patch_sharpness_var":float(np.var(sl)),"patch_saturation_var":float(np.var(sal))}

# --- JPEG Ghost [v4 NEW] ---
def jpeg_ghost_detector(image):
    img_rgb = image.convert("RGB"); img_arr = np.array(img_rgb).astype(np.float32)
    gi = []
    for q in [75,85,95]:
        buf = BytesIO(); img_rgb.save(buf,format="JPEG",quality=q); buf.seek(0)
        gi.append(float(np.mean(np.abs(img_arr-np.array(Image.open(buf).convert("RGB")).astype(np.float32)))))
    gi = np.array(gi)
    return {"jpeg_ghost_mean":float(np.mean(gi)),"jpeg_ghost_std":float(np.std(gi)),
            "jpeg_ghost_ratio":float(np.min(gi)/(np.max(gi)+1e-10))}

print("All 18 detectors loaded (12 unchanged + 3 new + 3 GPU).")

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP multi-prompt detector loaded (8 prompt pairs). [v4]
All 18 detectors loaded (12 unchanged + 3 new + 3 GPU).


## CELL 23: Combined Pipeline


In [12]:
import concurrent.futures

def cpu_detectors(image, full_image=None):
    result = {}
    for det in [fft_detector, ela_detector, noise_detector, metadata_detector,
                dct_detector, wavelet_detector, color_histogram_detector, lbp_detector,
                edge_coherence_detector, pixel_stats_detector, gan_fingerprint_detector,
                gradient_detector, jpeg_ghost_detector]:
        result.update(det(image))
    if full_image is not None:
        for det in [srm_detector, patch_consistency_detector]:
            result.update(det(full_image))
    else:
        for det in [srm_detector, patch_consistency_detector]:
            result.update(det(image))
    return result

def run_all_detectors_batched(original_img, full_img, transformations_dict):
    attack_names = list(transformations_dict.keys())
    transformed_images = [transformations_dict[a](original_img.copy()) for a in attack_names]
    transformed_full_images = [transformations_dict[a](full_img.copy()) for a in attack_names] if full_img is not None else [None]*len(attack_names)
    siglip_results = siglip_batch_detector(transformed_images)
    vit_results = vit_batch_detector(transformed_images)
    clip_results = clip_batch_detector(transformed_images)
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        cpu_results = list(ex.map(cpu_detectors, transformed_images, transformed_full_images))
    final_scores = []
    for i in range(len(attack_names)):
        scores = {}
        scores.update(siglip_results[i]); scores.update(vit_results[i])
        scores.update(clip_results[i]); scores.update(cpu_results[i])
        np_arr = [siglip_results[i].get("siglip_ai_prob",0.5),
                  vit_results[i].get("vit_ai_prob",0.5),
                  clip_results[i].get("clip_ai_prob",0.5)]
        scores["neural_ensemble_prob"] = float(np.mean(np_arr))
        scores["neural_ensemble_std"] = float(np.std(np_arr))
        scores["neural_ensemble_max"] = float(np.max(np_arr))
        scores["neural_ensemble_min"] = float(np.min(np_arr))
        final_scores.append(scores)
    return attack_names, final_scores

print("Batched pipeline ready (18 detectors + stacking features). [v4]")


# ==========================================================
# CELL 24: Process Real + AI Images
# (Same loop as v3, writing to v4 CSVs)
# ==========================================================

SAVE_INTERVAL = 100

def process_images(folder, csv_path, label, desc):
    SUPPORTED_EXTS = (".png",".jpg",".jpeg",".webp",".bmp",".avif")
    image_files = [f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS]
    print(f"{desc} folder: {folder}\nFound {len(image_files)} images")
    processed_ids = set()
    if Path(csv_path).exists():
        processed_ids = set(pd.read_csv(csv_path)["image_id"].astype(str).tolist())
    rows = []
    for image_path in tqdm(image_files, desc=desc):
        try:
            full_img = Image.open(image_path).convert("RGB")
            original = full_img.copy()
            original.thumbnail((1024, 1024))
        except Exception as e:
            tqdm.write(f"FAILED: {image_path} -- {e}"); continue
        attack_names, batch_scores = run_all_detectors_batched(original, full_img, transformations)
        for i, attack_name in enumerate(attack_names):
            image_id = str(image_path.relative_to(folder)).replace("/","_").replace("\\","_") + "_" + attack_name
            if image_id in processed_ids: continue
            if attack_name == "none": at,ast = "none","0"
            else: parts = attack_name.split("_"); at = parts[0]; ast = "_".join(parts[1:])
            row = {"image_id":image_id,"original_path":str(image_path),"label":label,"attack_type":at,"attack_strength":ast}
            row.update(batch_scores[i]); rows.append(row); processed_ids.add(image_id)
        if len(rows) >= SAVE_INTERVAL:
            temp_df = pd.DataFrame(rows)
            if Path(csv_path).exists(): temp_df.to_csv(csv_path,mode="a",header=False,index=False)
            else: temp_df.to_csv(csv_path,index=False)
            tqdm.write(f"Checkpoint saved ({len(rows)} rows)"); rows = []
    if rows:
        temp_df = pd.DataFrame(rows)
        if Path(csv_path).exists(): temp_df.to_csv(csv_path,mode="a",header=False,index=False)
        else: temp_df.to_csv(csv_path,index=False)
    result_df = pd.read_csv(csv_path)
    print(f"\nDONE -- {desc} (Shape: {result_df.shape})")
    return result_df

# Find folders
REAL_FOLDER = DATASET/"Real-images"/"Real-images"
if not REAL_FOLDER.exists(): REAL_FOLDER = DATASET/"Real-images"
AI_FOLDER = DATASET/"AI-images"/"AI-images"
if not AI_FOLDER.exists(): AI_FOLDER = DATASET/"AI-images"

real_df = process_images(REAL_FOLDER, REAL_CSV, 0, "Real Images")
ai_df = process_images(AI_FOLDER, AI_CSV, 1, "AI Images")

Batched pipeline ready (18 detectors + stacking features). [v4]
Real Images folder: /kaggle/input/datasets/ishu15m/ai-vs-real-images/Real-images/Real-images
Found 298 images



Real Images:   1%|          | 2/298 [00:24<1:01:37, 12.49s/it]
                                                           
Real Images:   1%|          | 3/298 [00:35<58:05, 11.82s/it]  

Checkpoint saved (132 rows)



Real Images:   2%|▏         | 5/298 [00:55<51:55, 10.63s/it]
                                                         
Real Images:   2%|▏         | 6/298 [01:05<51:00, 10.48s/it]

Checkpoint saved (132 rows)


...

Checkpoint saved (132 rows)


AI Images:  99%|█████████▉| 276/278 [3:41:15<01:19, 39.64s/it]

Checkpoint saved (132 rows)


AI Images: 100%|██████████| 278/278 [3:43:02<00:00, 48.14s/it]



DONE -- AI Images (Shape: (12232, 90))


## CELL 25: Training v4


In [15]:
import joblib, json
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_curve)

df = pd.concat([pd.read_csv(REAL_CSV), pd.read_csv(AI_CSV)], ignore_index=True)
df.to_csv(COMBINED_CSV, index=False)
df = df[df["attack_type"] == "none"]
df = df.dropna(axis=1, how="all")
for col in df.columns:
    if df[col].dtype == "object": df[col] = df[col].fillna("unknown")
    else: df[col] = df[col].fillna(df[col].median())
df["label"] = df["label"].astype(int)

detector_features = [
    "siglip_ai_prob","siglip_confidence","vit_ai_prob","vit_confidence",
    "fft_low_energy","fft_mid_energy","fft_high_energy","fft_high_freq_ratio","fft_entropy","fft_mid_to_high_ratio",
    "ela_mean_q95","ela_std_q95","ela_max_q95","ela_mean_q75","ela_std_q75","ela_skew","ela_kurtosis",
    "noise_std_gauss","noise_mean_gauss","noise_std_median","noise_laplacian_var","noise_channel_std_range","noise_channel_std_mean",
    "meta_has_icc_profile","meta_exif_field_count","meta_has_camera_make","meta_has_gps","meta_compression_ratio",
    "meta_has_alpha","meta_num_channels",
    "dct_block_energy","dct_block_std","dct_boundary_strength","dct_hf_coeff_ratio",
    "wavelet_detail_energy","wavelet_approx_energy","wavelet_detail_ratio","wavelet_hh_entropy","wavelet_hh_std",
    "color_entropy_r","color_entropy_g","color_entropy_b","color_corr_rg","color_corr_rb",
    "lbp_entropy","lbp_uniformity","lbp_mean","lbp_std",
    "edge_density","edge_dir_entropy","edge_dir_uniformity","edge_magnitude_std",
    "pixel_benford_dev","pixel_entropy","pixel_unique_ratio","pixel_dynamic_range","pixel_mean_brightness",
    "gan_autocorr_peak","gan_autocorr_mean","gan_periodicity","gan_spectral_flatness",
    "gradient_mean","gradient_std","gradient_kurtosis","gradient_high_ratio",
    "srm_noise_energy","srm_noise_std","srm_noise_skewness","srm_noise_kurtosis","srm_cross_channel_corr",
    "patch_fft_var","patch_noise_var","patch_sharpness_var","patch_saturation_var",
    "jpeg_ghost_mean","jpeg_ghost_std","jpeg_ghost_ratio",
]
features = [c for c in detector_features if c in df.columns]
print(f"Features ({len(features)})")

X = df[features]; y = df["label"]; groups = df["original_path"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train,X_test = X.iloc[train_idx],X.iloc[test_idx]
y_train,y_test = y.iloc[train_idx],y.iloc[test_idx]
assert len(set(groups.iloc[train_idx])&set(groups.iloc[test_idx]))==0

spw = (y_train==0).sum()/max((y_train==1).sum(),1)
xgb = XGBClassifier(n_estimators=800,max_depth=6,learning_rate=0.03,subsample=0.7,
    colsample_bytree=0.6,min_child_weight=5,gamma=0.1,reg_alpha=0.5,reg_lambda=2.0,
    scale_pos_weight=spw,eval_metric="logloss",random_state=42,nthread=1)
lgbm = LGBMClassifier(n_estimators=800,learning_rate=0.03,max_depth=6,num_leaves=31,
    min_child_samples=20,reg_alpha=0.5,reg_lambda=2.0,is_unbalance=True,random_state=42,verbose=-1,n_jobs=1)
rf = RandomForestClassifier(n_estimators=800,max_depth=15,min_samples_split=10,
    min_samples_leaf=5,class_weight="balanced",random_state=42,n_jobs=-1)
ensemble = VotingClassifier(estimators=[("xgb",xgb),("lgbm",lgbm),("rf",rf)],voting="soft",n_jobs=-1)

print("Training v4 ensemble..."); ensemble.fit(X_train, y_train); print("Done.")

y_prob = ensemble.predict_proba(X_test)[:, 1]
fpr,tpr,thresholds = roc_curve(y_test, y_prob)
optimal_threshold = thresholds[np.argmax(tpr-fpr)]
print(f"Optimal threshold: {optimal_threshold:.4f}")
y_pred = (y_prob >= optimal_threshold).astype(int)

print(f"\nAccuracy: {accuracy_score(y_test,y_pred):.4f}")
print(f"Precision: {precision_score(y_test,y_pred):.4f}")
print(f"Recall: {recall_score(y_test,y_pred):.4f}")
print(f"F1: {f1_score(y_test,y_pred):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test,y_pred)}")
print(f"\n{classification_report(y_test,y_pred,target_names=['Real(0)','AI(1)'])}")

# Feature importance check
imp = pd.Series(ensemble.named_estimators_["rf"].feature_importances_, index=features).sort_values(ascending=False)
print("\nTop 20 Features:"); print(imp.head(20).to_string())
meta_imp = imp[[f for f in imp.index if f.startswith("meta_")]]
print(f"\n[v4 CHECK] Max metadata importance: {meta_imp.max():.4f} (should be <0.05)")
neural_imp = imp[[f for f in imp.index if any(k in f for k in ["siglip","vit","clip","neural"])]]
print(f"[v4 CHECK] Total neural importance: {neural_imp.sum():.4f}")

# Save
joblib.dump(ensemble, os.path.join(MODEL_DIR, "ensemble_v4.joblib"))
with open(os.path.join(MODEL_DIR, "features_v4.json"), "w") as f: json.dump(features, f)
with open(os.path.join(MODEL_DIR, "optimal_threshold_v4.json"), "w") as f:
    json.dump({"optimal_threshold": float(optimal_threshold)}, f)

# Confidence tiers
def get_tier(p):
    if p<=0.15: return "Real(HIGH)"
    elif p<=0.35: return "Real(MED)"
    elif p<=0.50: return "Real(LOW)"
    elif p<=0.65: return "AI(LOW)"
    elif p<=0.85: return "AI(MED)"
    else: return "AI(HIGH)"

results = pd.DataFrame({"image_id":df["image_id"].iloc[test_idx].values,
    "original_path":df["original_path"].iloc[test_idx].values,
    "actual":y_test.values,"predicted":y_pred,"probability_ai":y_prob})
results["confidence_tier"] = results["probability_ai"].apply(get_tier)
fp = results[(results["actual"]==0)&(results["predicted"]==1)]
fn = results[(results["actual"]==1)&(results["predicted"]==0)]
print(f"\nFP: {len(fp)}, FN: {len(fn)}, Total Errors: {len(fp)+len(fn)}")
print(f"\nConfidence tiers:\n{results['confidence_tier'].value_counts().to_string()}")

errors = pd.concat([fp.assign(error_type="FP"), fn.assign(error_type="FN")])
errors.to_csv(os.path.join(PROJECT_DIR, "fp_fn_images_v4.csv"), index=False)

print("\n" + "="*50 + "\nPIPELINE v4 COMPLETE\n" + "="*50)

Features (77)
Training v4 ensemble...
Done.
Optimal threshold: 0.6314

Accuracy: 0.9828
Precision: 0.9833
Recall: 0.9833
F1: 0.9833

Confusion Matrix:
[[55  1]
 [ 1 59]]

              precision    recall  f1-score   support

     Real(0)       0.98      0.98      0.98        56
       AI(1)       0.98      0.98      0.98        60

    accuracy                           0.98       116
   macro avg       0.98      0.98      0.98       116
weighted avg       0.98      0.98      0.98       116


Top 20 Features:
meta_has_icc_profile      0.144397
wavelet_hh_entropy        0.080925
fft_entropy               0.074670
jpeg_ghost_ratio          0.064014
ela_mean_q95              0.043029
dct_block_energy          0.040005
siglip_ai_prob            0.035431
jpeg_ghost_mean           0.035048
pixel_unique_ratio        0.032584
ela_std_q75               0.023986
ela_mean_q75              0.021545
patch_fft_var             0.019725
meta_compression_ratio    0.019070
fft_high_energy           0.0

In [19]:
## CELL 26: Interactive Test (TTA Inference)
def test_image_tta(image_path, model, features_list):
    try:
        full_img = Image.open(image_path).convert('RGB')
        original = full_img.copy()
        original.thumbnail((1024, 1024))
    except Exception as e:
        print(f'Error loading {image_path}: {e}')
        return
        
    print('Running 44 transforms and detectors...')
    attack_names, batch_scores = run_all_detectors_batched(original, full_img, transformations)
    
    # Create DataFrame
    df_test = pd.DataFrame(batch_scores)
    # Filter only required features
    cols = [c for c in features_list if c in df_test.columns]
    X_test = df_test[cols].copy()
    for col in X_test.columns:
        X_test[col] = X_test[col].fillna(X_test[col].median() if not X_test[col].isnull().all() else 0)
        
    probs = model.predict_proba(X_test)[:, 1]
    avg_prob = np.mean(probs)
    
    if 0.4 <= avg_prob <= 0.6:
        decision = 'Inconclusive'
    elif avg_prob < 0.5:
        decision = 'Real'
    else:
        decision = 'AI'
        
    print(f'TTA Average Probability (AI): {avg_prob:.4f} -> {decision}')
    return avg_prob, decision

# Example usage:
# test_image_tta('path/to/image.png', ensemble, features)
#test_image_tta("/kaggle/input/datasets/ishu15m/ai-test1/ChatGPT Image Jun 18 2026 02_22_40 PM.png", ensemble, features)
test_image_tta("/kaggle/input/datasets/ishu15m/ai-test1/ChatGPT Image Jun 18 2026 02_25_48 PM.png", ensemble, features)

Running 44 transforms and detectors...
TTA Average Probability (AI): 0.6799 -> AI


(np.float64(0.6798932023521287), 'AI')

In [29]:
## CELL 26: Interactive Test (TTA Inference)
from PIL import Image
import pandas as pd
import numpy as np

def test_image_tta(image_path, model, features_list):
    image_path = image_path.strip('\'"')
    
    try:
        full_img = Image.open(image_path).convert('RGB')
        original = full_img.copy()
        original.thumbnail((1024, 1024))
    except Exception as e:
        print(f'Error loading {image_path}: {e}')
        return
        
    print(f'Running transforms and detectors on: {image_path}...')
    attack_names, batch_scores = run_all_detectors_batched(original, full_img, transformations)
    
    # Create DataFrame
    df_test = pd.DataFrame(batch_scores)
    # Filter only required features
    cols = [c for c in features_list if c in df_test.columns]
    X_test = df_test[cols].copy()
    for col in X_test.columns:
        X_test[col] = X_test[col].fillna(X_test[col].median() if not X_test[col].isnull().all() else 0)
        
    probs = model.predict_proba(X_test)[:, 1]
    avg_prob = np.mean(probs)
    
    # ----------------------------------------------------
    # UPDATED LOGIC: Using 0.5 for TTA, with an Inconclusive zone
    # ----------------------------------------------------
    if 0.45 <= avg_prob <= 0.55:
        decision = 'Inconclusive (Borderline)'
    elif avg_prob > 0.55:
        decision = 'AI'
    else:
        decision = 'Real'
    
    # Determine confidence tier
    if avg_prob >= 0.85 or avg_prob <= 0.15:
        tier = "HIGH"
    elif avg_prob >= 0.65 or avg_prob <= 0.35:
        tier = "MED"
    else:
        tier = "LOW"
        
    print("-" * 45)
    print(f'Prediction:      {decision}')
    print(f'Average AI Prob: {avg_prob:.4f}')
    print(f'Confidence Tier: ({tier})')
    print("-" * 45)
    
    return avg_prob, decision

# --- INTERACTIVE PROMPT ---
user_path = input("Enter the path to the image: ")

if user_path.strip():
    test_image_tta(user_path, ensemble, features)
else:
    print("No path entered.")


Enter the path to the image:  /kaggle/input/datasets/ishu15m/real-test1/pexels-manishjangid-30195652.jpg


Running transforms and detectors on: /kaggle/input/datasets/ishu15m/real-test1/pexels-manishjangid-30195652.jpg...
---------------------------------------------
Prediction:      Inconclusive (Borderline)
Average AI Prob: 0.4594
Confidence Tier: (LOW)
---------------------------------------------
